# Practical Assignment 5: CNN on Pneumonia Chest X-Ray Dataset

**Aim:** Build an image classification model using CNN on the Pneumonia Chest X-Ray dataset (Binary Classification: Normal vs Pneumonia).

**Key Concepts to Remember:**
1. **Image Loader:** `tf.keras.utils.image_dataset_from_directory` loads images from disk without manual looping.
2. **Rescaling Layer:** `Rescaling(1./255)` handles normalization directly inside the neural network.
3. **Binary/Multiclass:** 2 classes can be classified using `Dense(2, 'softmax')` with `sparse_categorical_crossentropy`.


In [1]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
import tensorflow as tf
import numpy as np

# Step 1: Load Chest X-Ray Images from train and test directories
train_dir = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/Pneumonia X Ray/train'
test_dir = r'd:/workspace01/4th year/DL/LabPracticesIV_DEEP_LEARNING/dataset/Pneumonia X Ray/test'

IMG_SIZE = (64, 64)
BATCH_SIZE = 32

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
).take(30)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False
).take(10)

class_names = ['NORMAL', 'PNEUMONIA']
print('Class Labels:', class_names)

# Step 2: Build CNN Architecture
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(64, 64, 3)),
    tf.keras.layers.Rescaling(1.0 / 255.0),
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.25),
    tf.keras.layers.Dense(2, activation='softmax')
])
model.summary()

# Step 3: Compile and Train Model
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(train_ds, epochs=3, validation_data=test_ds, verbose=1)

# Step 4: Evaluate and Predict
loss, acc = model.evaluate(test_ds, verbose=0)
print(f'\nPneumonia Detection Test Accuracy: {acc*100:.2f}% | Test Loss: {loss:.4f}')

for images, labels in test_ds.take(1):
    preds = model.predict(images[:3], verbose=0)
    for i in range(3):
        pred_c = class_names[np.argmax(preds[i])]
        true_c = class_names[labels[i]]
        print(f'Sample {i+1} -> Predicted: {pred_c:<10} | Actual: {true_c}')


Found 3937 files belonging to 2 classes.
Found 624 files belonging to 2 classes.
Class Labels: ['NORMAL', 'PNEUMONIA']
Model: "sequential_4"
┌─────────────────────────────────┬────────────────────────┬───────────────┐
│ Layer (type)                    │ Output Shape           │       Param # │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ rescaling (Rescaling)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 62, 62, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 31, 31, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 29, 29, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_poolin